### Load the preprocessed data

In [1]:
import pandas as pd

X_train = pd.read_csv('dataset/X_train.csv')
y_train = pd.read_csv('dataset/y_train.csv').squeeze()

X_test = pd.read_csv('dataset/X_test.csv')
y_test = pd.read_csv('dataset/y_test.csv').squeeze()

X_train_balanced = pd.read_csv('dataset/X_train_balanced.csv')
y_train_balanced = pd.read_csv('dataset/y_train_balanced.csv').squeeze()

print(
    "Raw train:", X_train.shape,
    "| Balanced train:", X_train_balanced.shape,
    "| Test:", X_test.shape
)

Raw train: (25217, 47) | Balanced train: (39544, 47) | Test: (6305, 47)


### Build a shared results table

In [2]:
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import cross_val_score

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

results = []


def evaluate_and_log(name, model, X_tr, y_tr, X_te, y_te, cv):

    cv_auc = cross_val_score(
        model,
        X_tr,
        y_tr,
        cv=cv,
        scoring='roc_auc',
        n_jobs=-1
    )

    model.fit(X_tr, y_tr)

    y_pred = model.predict(X_te)
    y_proba = model.predict_proba(X_te)[:, 1]

    row = {
        'model': name,
        'cv_roc_auc_mean': round(cv_auc.mean(), 3),
        'test_accuracy': round(accuracy_score(y_te, y_pred), 3),
        'test_precision': round(precision_score(y_te, y_pred), 3),
        'test_recall': round(recall_score(y_te, y_pred), 3),
        'test_f1': round(f1_score(y_te, y_pred), 3),
        'test_roc_auc': round(roc_auc_score(y_te, y_proba), 3)
    }

    results.append(row)

    print(row)

    return model

### Train the Logistic Regression baseline

In [3]:
from sklearn.linear_model import LogisticRegression

lr_baseline = evaluate_and_log(
    'Logistic Regression (baseline, raw)',
    LogisticRegression(
        max_iter=1000,
        random_state=42
    ),
    X_train,
    y_train,
    X_test,
    y_test,
    cv
)

{'model': 'Logistic Regression (baseline, raw)', 'cv_roc_auc_mean': np.float64(0.851), 'test_accuracy': 0.849, 'test_precision': 0.73, 'test_recall': 0.477, 'test_f1': 0.577, 'test_roc_auc': 0.856}


### Train on SMOTE-balanced data and compare

In [4]:
lr_balanced = evaluate_and_log(
    'Logistic Regression (baseline, SMOTE-balanced)',
    LogisticRegression(
        max_iter=1000,
        random_state=42
    ),
    X_train_balanced,
    y_train_balanced,
    X_test,
    y_test,
    cv
)

{'model': 'Logistic Regression (baseline, SMOTE-balanced)', 'cv_roc_auc_mean': np.float64(0.91), 'test_accuracy': 0.793, 'test_precision': 0.517, 'test_recall': 0.615, 'test_f1': 0.562, 'test_roc_auc': 0.81}


### Hyperparameter tuning

In [5]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'C': [0.01, 0.1, 1, 10, 100],
    'solver': ['liblinear', 'lbfgs'],
    'class_weight': [None, 'balanced']
}

lr_grid = GridSearchCV(
    LogisticRegression(
        max_iter=1000,
        random_state=42
    ),
    param_grid,
    cv=cv,
    scoring='roc_auc',
    n_jobs=-1
)

lr_grid.fit(X_train, y_train)

print("Best params:", lr_grid.best_params_)
print("Best CV ROC-AUC:", round(lr_grid.best_score_, 3))

Best params: {'C': 0.1, 'class_weight': 'balanced', 'solver': 'lbfgs'}
Best CV ROC-AUC: 0.852


### Best parameters and tuned model

In [6]:
lr_tuned = evaluate_and_log(
    'Logistic Regression (tuned)',
    lr_grid.best_estimator_,
    X_train,
    y_train,
    X_test,
    y_test,
    cv
)

{'model': 'Logistic Regression (tuned)', 'cv_roc_auc_mean': np.float64(0.852), 'test_accuracy': 0.781, 'test_precision': 0.496, 'test_recall': 0.772, 'test_f1': 0.604, 'test_roc_auc': 0.856}


### Compare tuned vs. baseline

In [7]:
lr_results = pd.DataFrame(results)

lr_results

,model,cv_roc_auc_mean,test_accuracy,test_precision,test_recall,test_f1,test_roc_auc
0,"Logistic Regression (baseline, raw)",0.851,0.849,0.730,0.477,0.577,0.856
1,"Logistic Regression (baseline, SMOTE-balanced)",0.910,0.793,0.517,0.615,0.562,0.810
2,Logistic Regression (tuned),0.852,0.781,0.496,0.772,0.604,0.856


### Model Comparison

In [9]:
results_df = pd.DataFrame(results)

results_df

,model,cv_roc_auc_mean,test_accuracy,test_precision,test_recall,test_f1,test_roc_auc
0,"Logistic Regression (baseline, raw)",0.851,0.849,0.730,0.477,0.577,0.856
1,"Logistic Regression (baseline, SMOTE-balanced)",0.910,0.793,0.517,0.615,0.562,0.810
2,Logistic Regression (tuned),0.852,0.781,0.496,0.772,0.604,0.856


In [10]:
results_df.sort_values(
    by='test_roc_auc',
    ascending=False
)

,model,cv_roc_auc_mean,test_accuracy,test_precision,test_recall,test_f1,test_roc_auc
0,"Logistic Regression (baseline, raw)",0.851,0.849,0.730,0.477,0.577,0.856
2,Logistic Regression (tuned),0.852,0.781,0.496,0.772,0.604,0.856
1,"Logistic Regression (baseline, SMOTE-balanced)",0.910,0.793,0.517,0.615,0.562,0.810
